# Lab 01 · Prompting and small evaluations

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · ~15–20 min core work (estimate; setup and extras take longer)

## Prompts, memory and evaluation
**Start with a problem:** If every question repeats the assistant’s instructions, changing those instructions means editing each prompt. A follow-up may also lack the previous answer.

**In this lab:** Separate reusable system instructions from user questions, carry a previous response forward, and check the result.

**Goal:** Learn to send system and user prompts, reuse a previous response, validate an answer, compare prompting and reasoning modes, and experiment with generation settings.

ShopEasy has web and mobile clients. An **API gateway** can provide shared routing and security at the edge; a **backend for frontend (BFF)** tailors an API to a particular client. A system can use both. We will use these ideas in short architecture questions, then try two small reasoning puzzles.

The model can make mistakes. Compare the actual replies and check them independently.

**At a glance**

![Lab 01 at a glance: Architecture question (gateway or BFF), then Prompt (system + user), then Remember (pass prior answer), then Validate (JSON + decision), then Compare (older + thinking modes).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/01.svg)

## Think about the example

An API gateway can route requests and apply common policies. A backend for frontend (BFF) can tailor an API to one client, such as a mobile app. A system may use both.

**Concepts:** API gateway; Backend for frontend (BFF); Client-specific APIs.

Use the scenario above to consider these questions before running the code:

1. What shared needs can an API gateway address?
2. Why might mobile and web clients benefit from different BFFs?

## The idea in context

A system message guides the assistant; a user message requests a task. A follow-up can include a previous answer as temporary context.

Parse and validate replies before using them. An older model and DeepSeek thinking modes can be compared on small, checkable examples.

| Key term | Meaning |
| --- | --- |
| **System prompt** | Instructions that guide the assistant’s task and response. |
| **User prompt** | The specific question or request sent by the user. |
| **Temporary context** | Information, such as a previous answer, sent again in a later request. |
| **Validation** | Checking that a response has the expected format and meets a specific condition. |
| **Thinking mode** | A model setting that enables additional internal reasoning before the final answer. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Save `DEEPSEEK_API_KEY` in Colab Secrets and keep `PROVIDER="deepseek"`. Step 4 downloads a small local model; Step 5 uses DeepSeek thinking mode. Other providers work in Steps 1–3; Step 5 requires DeepSeek. Each client stops after 12 calls.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.0" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/fc798d4d9fda69756f7d860756fe7c0507f7f2a2.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
PROVIDER = "deepseek"  # or "gemini" or "freeinference" for Steps 1–3
from coursekit.runtime import Client, secret
from coursekit.notebook import PROVIDERS
settings = PROVIDERS[PROVIDER]
key = secret(settings["secret_name"])
plain = Client(model=settings["model"], base_url=settings["base_url"],
               api_key=key, max_calls=12, extra_body=settings.get("extra_body"))
thinking = None
if PROVIDER == "deepseek":
    thinking = Client(model=settings["model"], base_url=settings["base_url"],
                      api_key=key, max_calls=12, max_tokens=4096, timeout=120,
                      extra_body={"thinking": {"type": "enabled"}, "reasoning_effort": "low"})

## Step 1 · From one mixed prompt to two roles
Start with **one mixed user prompt**: the assistant's role, response style, and the architecture question are all in the same text. It can work, but it is harder to maintain when the question changes because the shared instructions and the changing request are mixed together.

Next, split the *same* content into two messages:
- **System prompt:** stable instructions for the assistant, such as its role and response style.
- **User prompt:** the current task and its specific facts or question.

The split makes shared instructions easier to reuse and the question easier to change. On services that support prefix caching, keeping a long shared prefix unchanged **may** reduce repeated input processing and cost. [DeepSeek enables context caching automatically](https://api-docs.deepseek.com/guides/kv_cache/), but hits are best-effort and depend on an eligible matching prefix. This tiny example does **not** measure or guarantee a cache hit; stable text in a user message can also be cached. System instructions do not guarantee correct or safe answers.

In [ ]:
mixed_prompt = """You are a software architecture tutor. Answer in one sentence.
ShopEasy needs a tailored API for its mobile app. What pattern fits?"""
mixed_answer = plain.ask(mixed_prompt, system="")

system_prompt = "You are a software architecture tutor. Answer in one sentence."
user_prompt = "ShopEasy needs a tailored API for its mobile app. What pattern fits?"
split_answer = plain.ask(user_prompt, system=system_prompt)

new_question = "For several services, which provides one routing entry point: an API gateway or a BFF?"
next_answer = plain.ask(new_question, system=system_prompt)
print("Mixed prompt:", mixed_answer)
print("Split prompt:", split_answer)
print("Same system prompt, new question:", next_answer)

## How temperature, top-k, and top-p work together

The picture follows one next-token decision in a common sampling pipeline: temperature adjusts scores, top-k keeps the highest-ranked choices, top-p narrows them by cumulative probability, and the model samples from what remains. The probabilities shown are illustrative.

![Sampling example: four scores become probabilities after temperature 2. Top-k 3 removes D; top-p 0.80 keeps A and B; the model then samples A with 62.25% or B with 37.75%.](https://mkassaf.github.io/agentic-ai-course/assets/images/sampling-settings.svg)

This is the order used by [Transformers for these settings](https://github.com/huggingface/transformers/blob/main/src/transformers/generation/utils.py) when sampling without other filters. Providers can apply settings differently or ignore them. The diagram uses `temperature=2`, `top_k=3`, and `top_p=0.80` for illustration; the exercises below use different values. Temperature zero is handled separately.

## Step 1a · Optional: explore the sampling simulator

Move the temperature, top-k, and top-p sliders to see which candidate tokens remain and how their probabilities change. Use **Post-Temp** to inspect the probabilities before the filters, and **Final Odds** to see the distribution after them. Try changing one setting at a time.

The simulator uses illustrative probabilities; it does not call an LLM or need a key. It applies temperature → top-k → top-p, then normalizes the remaining choices. Providers can use different rules. The code below embeds it in Colab; if the frame does not load, [open the simulator in a new tab](https://mkassaf.github.io/agentic-ai-course/assets/tools/sampling.html).

In [ ]:
from IPython.display import IFrame, display

display(IFrame("https://mkassaf.github.io/agentic-ai-course/assets/tools/sampling.html",
               width="100%", height=1100))

## Step 1b · Optional: change the temperature

A model chooses its next token from possible continuations. **Temperature** changes how strongly it favors higher-probability choices: a lower value generally makes the distribution more concentrated; a higher value makes it flatter. It does not directly control factual accuracy, and a low value does not guarantee identical answers.

Run setup and Step 1 first. Keep the same architecture question, system instruction, model, and output limit. Make one live request at `temperature=0.0` and another at `1.3`. Compare the answers, then rerun to see whether they change. Two answers may be identical; two samples cannot establish which setting is better.

`extra_body` supplies the model's API parameters. The existing provider options are preserved, including DeepSeek's disabled-thinking setting. Each client has its own one-call limit; these are two additional model requests. Use a value supported by your selected provider. [DeepSeek documents temperature from 0 to 2](https://api-docs.deepseek.com/api/create-chat-completion/) and says it has no effect in thinking mode; this example uses thinking disabled.

In [ ]:
config_question = "ShopEasy has web and mobile clients. In one sentence, suggest one trade-off of adding a BFF."
low_options = {**(settings.get("extra_body") or {}), "temperature": 0.0}
high_options = {**(settings.get("extra_body") or {}), "temperature": 1.3}

low = Client(model=settings["model"], base_url=settings["base_url"], api_key=key,
             max_calls=1, max_tokens=128, extra_body=low_options)
high = Client(model=settings["model"], base_url=settings["base_url"], api_key=key,
              max_calls=1, max_tokens=128, extra_body=high_options)
print("Temperature 0.0:", low.ask(config_question, system=system_prompt))
print("Temperature 1.3:", high.ask(config_question, system=system_prompt))

## Step 2 · Temporary memory: pass the previous answer
First ask a follow-up without context. Then save one response in `first_answer` and include it in the next user prompt. The second call can use that text as context. This is temporary memory held in a Python variable: each API request is independent, and the model does not remember the previous request unless we send the context again. It is not persistent memory.

![Temporary memory: without context, a follow-up does not identify the choice. With context, save the first model answer in first_answer and include it in the second request along with the follow-up question.](https://mkassaf.github.io/agentic-ai-course/assets/images/temporary-memory.svg)

Because this version copies the response into a new **user message**, it is not a complete role-labelled chat history. A production conversation may instead resend selected user and assistant messages as separate roles.

In [ ]:
without_context = plain.ask("What is one drawback of this choice? Answer in one sentence.")

first_answer = plain.ask(
    "ShopEasy's mobile app needs a tailored API. Suggest one pattern in one sentence.")

follow_up = f"""Earlier answer: {first_answer}
What is one drawback of this choice? Answer in one sentence."""
second_answer = plain.ask(follow_up)
print("Without context:", without_context)
print("First answer:", first_answer)
print("Follow-up:", second_answer)

## Step 3 · Validate a structured answer
Ask for one JSON object with a decision and a reason. We then check both the **format** and a small expected answer. The scenario explicitly says each client needs its own API, so our classroom answer is `BFF`.

`json.loads` converts JSON text to a Python value. A valid format or matching label does not prove that the reason is correct; read it yourself. The `try` block keeps malformed JSON from crashing the notebook.

In [ ]:
import json

prompt = """Web and mobile clients need different tailored APIs.
Choose BFF or API gateway. Return JSON only, with keys
"decision" and "reason". Give a short reason."""
reply = plain.ask(prompt)
print("Model reply:", reply)
try:
    answer = json.loads(reply)
    valid = (isinstance(answer, dict) and answer.get("decision") in ["BFF", "API gateway"]
             and isinstance(answer.get("reason"), str) and bool(answer["reason"].strip()))
    correct = valid and answer["decision"] == "BFF"
    print("Valid format:", valid, "| Expected decision:", correct)
except json.JSONDecodeError:
    print("Valid format: False | The reply is not JSON")

## Step 4 · See a small model make mistakes
Use the older, small [SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) model on two questions. First ask directly; then try a guided prompt that suggests steps (a simple chain-of-thought prompting technique). Python independently prints the correct answers, so you can compare them with the model's actual replies.

In one test, the direct answers said there was **1** “r” in strawberry and that **9.9 and 9.11 are equal**. Both answers are wrong. Your results may differ with model and library versions; a non-thinking model can sometimes answer correctly. The guided prompts may help, but they can also be wrong or give an incorrect explanation even when the final answer is right.

This step downloads the model and runs it on the Colab CPU. The first run needs internet and may take several minutes. `9.9` and `9.11` are decimal numbers, not software versions.

In [ ]:
if find_spec("transformers") is None:
    %pip install -q "transformers>=4.45,<5" torch
from transformers import pipeline
from decimal import Decimal

old = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", device=-1)
def ask_old(prompt):
    result = old([{"role": "user", "content": prompt}], max_new_tokens=96, do_sample=False)
    return result[0]["generated_text"][-1]["content"]

letters = "How many times does the letter 'r' appear in strawberry? Reply with only a number."
decimals = ("Compare the decimal numbers 9.9 and 9.11. "
            "Which is greater? Reply with only the greater decimal.")
guided_letters = ("How many r's are in strawberry? Think step by step, "
                  "list the letters in the word, and count the r's. Finish with the number.")
guided_decimals = ("Which decimal is larger, 9.9 or 9.11? Step 1: write both with two digits "
                   "after the decimal point. Step 2: compare the values. Then answer.")
print("Strawberry — direct:", ask_old(letters))
print("Strawberry — guided:", ask_old(guided_letters))
print("Correct count (Python):", "strawberry".count("r"))
print("Decimals — direct:", ask_old(decimals))
print("Decimals — guided:", ask_old(guided_decimals))
print("Correct larger decimal (Python):", max(Decimal("9.9"), Decimal("9.11")))

## Step 4a · Optional: change top-k

**Top-k** limits the next-token choices to the `k` highest-probability candidates. With `top_k=1`, the local model selects its highest-probability token at each step; with `top_k=20`, it may sample among a wider set. **Top-p** instead keeps a variable-sized set whose cumulative probability reaches a chosen threshold. `max_new_tokens` limits response length and may cut an answer off.

The picture above walks through a numerical example of the two filters.

This example uses the local SmolLM model loaded in Step 4, not DeepSeek. Hugging Face sampling requires `do_sample=True`; we keep `temperature=0.8` and `top_p=1.0` the same and change only `top_k`. Run Step 4 first. Resetting the random seed makes the comparison easier to repeat in the same environment; outputs can still match or be wrong, and results can vary across environments. This tiny model may produce irrelevant or incorrect architecture advice; inspect its answers. [Hugging Face generation settings](https://huggingface.co/docs/transformers/main_classes/text_generation)

DeepSeek's [Chat Completions API](https://api-docs.deepseek.com/api/create-chat-completion/) documents `temperature` and `top_p`, but not `top_k`. For the default `deepseek-flash` model, `top_p` is fixed at 1.0 when thinking is disabled, so changing `top_p` has no effect in that mode. Other providers and API endpoints can differ; check their documentation before sending a parameter.

In [ ]:
from transformers import set_seed

top_k_question = "Choose a way to handle a slow Inventory service: retry, timeout, or circuit breaker. Answer with one choice."

def sample_top_k(k):
    result = old([{"role": "user", "content": top_k_question}], max_new_tokens=40,
                 do_sample=True, temperature=0.8, top_k=k, top_p=1.0)
    return result[0]["generated_text"][-1]["content"]

set_seed(42)
print("Top-k 1:", sample_top_k(1))
set_seed(42)
print("Top-k 20:", sample_top_k(20))

## Step 5 · Compare DeepSeek without and with thinking
Ask the same two direct questions with DeepSeek's **thinking disabled**, then **thinking enabled**. We do not ask the thinking model to reveal a chain of thought; it can use its own internal reasoning. The helper returns only the final answer.

Both calls use `deepseek-flash`: this is a comparison of two modes of the same model, not an old-versus-new model comparison. The older SmolLM2 experiment above changes the model as well, so differences cannot be attributed to reasoning mode alone. Reasoning mode may use more tokens and time; it can also be wrong. Test rather than assume it always wins.

The thinking-mode API settings below are specific to DeepSeek. If you selected Gemini or FreeInference for Steps 1–3, switch `PROVIDER` to `"deepseek"` and rerun setup to run this step. Use a DeepSeek API key in Colab Secrets.

This step repeats the two questions, so it can run even if the local model in Step 4 is unavailable.

In [ ]:
from decimal import Decimal

letters = "How many times does the letter 'r' appear in strawberry? Reply with only a number."
decimals = "Compare the decimal numbers 9.9 and 9.11. Which is greater? Reply with only the greater decimal."

if thinking is None:
    print("Switch PROVIDER to deepseek in setup to run the thinking comparison.")
else:
    print("DeepSeek, no thinking:", plain.ask(letters))
    print("DeepSeek, thinking:", thinking.ask(letters))
    print("DeepSeek, no thinking:", plain.ask(decimals))
    print("DeepSeek, thinking:", thinking.ask(decimals))
print("Python checks: r count =", "strawberry".count("r"))
print("Python checks: 9.9 > 9.11 is", Decimal("9.9") > Decimal("9.11"))

## What to look for
Compare the direct and guided answers, check the JSON response, and use Python's two checks to verify the puzzles. Look for the small model’s incorrect direct answers, but report what it actually says if your results differ. Compare the guided answer and its reasoning with the Python checks. Either DeepSeek mode can also be right or wrong. In the optional settings examples, compare responses without assuming that more variation means better answers.

## Try it
1. In Step 1, keep the system prompt the same and write another architecture question. Which parts can be reused?
2. Remove the earlier answer from `follow_up` in Step 2. What context is missing?
3. After the model responds, replace `reply` with a JSON answer whose decision is `API gateway`, then rerun the validation. Which check fails?
4. Change one sampling setting at a time in the optional examples. Do repeated responses always change?
5. Report the actual answers from both DeepSeek modes. Did the guided older-model prompt help? Repeat if you have time; these two examples are not a benchmark.

## Takeaway
A system message sets behavior, a user message asks for a task, and temporary context must be sent again. Validate outputs and test reasoning claims rather than assuming a model or prompt is always right.